DEPENDICIES

In [13]:
pip install -r requirements.txt -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm
import timm



c:\Users\maxle\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
#for nvidia GPU    NOT FULLY WORKING YET ALSO NEED TO ADD APPLE CHIP
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'using device: {device}')

using device: cpu


In [ ]:
batchsize = 16
workers = 2


arcs = 64
arcm = 0.5

GATHER DATASETS

In [ ]:
testdf = pd.read_csv('data/test.csv')
traindf = pd.read_csv('data/train.csv')

traindir = Path('data/train')
testdir = Path('data/test')

EDA

In [ ]:
print('train columns: ',traindf.columns.to_list())
print(traindf.head())
print('+++++++++++++++++++++++++++++++++++++++++++++')
print(f'Total training images: {len(traindf)}')

print(f'unique jaguars: {traindf['ground_truth'].nunique()}')
print('+++++++++++++++++++++++++++++++++++++++++++++')
print('class blanace:')
counts = traindf['ground_truth'].value_counts()
print(f'Images per Jaguar:')
print(f'min: {counts.min()}')
print(f'max: {counts.max()}')
print(f'mean: {counts.mean()}')

print('\ntest structure:')
print(testdf.head())
print(f'total pairs: ')
print(len(testdf))

train columns:  ['filename', 'ground_truth']
         filename ground_truth
0  train_0001.png        Abril
1  train_0002.png        Abril
2  train_0003.png        Abril
3  train_0004.png       Akaloi
4  train_0005.png       Akaloi
+++++++++++++++++++++++++++++++++++++++++++++
Total training images: 1895
unique jaguars: 31
+++++++++++++++++++++++++++++++++++++++++++++
class blanace:
Images pre Jaguar:
min: 13
max: 183
mean: 61.12903225806452

test structure:
   row_id    query_image  gallery_image
0       0  test_0001.png  test_0002.png
1       1  test_0001.png  test_0003.png
2       2  test_0001.png  test_0004.png
3       3  test_0001.png  test_0005.png
4       4  test_0001.png  test_0006.png
total pairs: 
137270


setting up trnasformers

In [ ]:
#applying randomness to avoid overfitting to promote learing of jaguar features
traintransform = transforms.Compose([
    transforms.RandomResizedCrop(348, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
    transforms.RandomGrayscale(p=.1), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std = [0.229, 0.224, 0.225])
])
valtransform = transforms.Compose([
    transforms.Resize(448),
    transforms.CenterCrop(448),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std = [0.229, 0.224, 0.225])
])

encodes the names of each jaguar to ints alphabetically
'abril' = 0
'akaloi' = 1
so on 

In [ ]:
class jaguardataset(Dataset):
    def __init__ (self, df: pd.DataFrame, dir: Path, transform=None):
        self.df = df.reset_index(drop=True)
        self.dir = dir
        self.transform = transform
        idx = sorted(self.df['ground_truth'].unique())
        self.labelmap = {v: i for i, v in enumerate(idx)}
        
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row =self.df.iloc[idx]
        img = Image.open(self.dir / row['filename']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = self.labelmap_map[row['ground_truth']]
        return img, label
    
from sklearn.model_selection import train_test_split

trainsplit, valsplit = train_test_split(
    traindf, test_size=0.2,
    stratify=traindf['ground_truth'],
    random_state=99
)

trainds = jaguardataset(trainsplit, traindir, traintransform)
valds = jaguardataset(valsplit, traindir, valtransform)
    
    

weighted sampler because some jaguars have over 100 images when other only have a few

In [ ]:
from torch.utils.data import WeightedRandomSampler

counts = trainsplit['ground_truth'].val_counts()
classweight = 1 = counts
sampleweight = trainsplit['ground_truth'].map(classweight).values
sampler = WeightedRandomSampler(
    weights=sampleweight,
    num_samples=len(sampleweight),
    replacement=True
)

LOADER

In [ ]:
Trainloader = DataLoader(trainds, batch_size=batchsize, sampler=sampler,num_workers=workers, pin_memory=True)
valloader = DataLoader(valds, batch_size=batchsize, shuffle=False ,num_workers=workers, pin_memory=True)

CREATE SUBMISSION

In [ ]:
submission = pd.DataFrame({
    'row_id': testdf['row_id'],
    'similarity': similarities
})

# Validate
assert len(submission) == 137270
assert submission['row_id'].tolist() == list(range(137270))
assert (submission['similarity'] >= 0).all()
assert (submission['similarity'] <= 1).all()

print("\nSubmission statistics:")
print(submission['similarity'].describe())

VALIDATION AND SUBMISSION

In [ ]:
validate_submission('my_submission.csv')

# Save submission
submission.to_csv('my_submission.csv', index=False)
print("\nSubmission saved: my_submission.csv")
print(f"File size: {Path('my_submission.csv').stat().st_size / 1024 / 1024:.2f} MB")